# Evaluate TrustedSQL

Notebook nay gom hai pha rieng: runtime va automatic evaluation. Metrics duoc tach rieng de de quan sat.

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import os
import subprocess
import sys
import yaml

PROJECT_ROOT = Path(os.environ.get('TRUSTEDSQL_PROJECT_ROOT', Path.cwd())).resolve()
EXPERIMENT_FILE = PROJECT_ROOT / 'configs' / 'experiments' / 'ex2_baseline_comparison.yaml'
EXPERIMENT_RUN_ID = f"ex2__{datetime.now().strftime('%Y%m%d_%H%M%S')}"
SYSTEMS = None
PROVIDERS = ['gemini_25_flash']
MAX_SAMPLES = None

EXPERIMENT_DIR = PROJECT_ROOT / 'outputs' / 'experiments' / EXPERIMENT_RUN_ID
PROJECT_ROOT, EXPERIMENT_RUN_ID, EXPERIMENT_DIR

## 0. Experiment Config Summary

Cell nay hien thi ro experiment, dataset, systems, providers va lenh se chay de trace thi nghiem.

In [ ]:
def read_yaml(path):
    with Path(path).open('r', encoding='utf-8-sig') as handle:
        return yaml.safe_load(handle) or {}


experiment_doc = read_yaml(EXPERIMENT_FILE)
experiment = experiment_doc['experiment']
dataset_profile = PROJECT_ROOT / experiment['dataset_profile']
dataset_doc = read_yaml(dataset_profile)
systems_dir = PROJECT_ROOT / 'configs' / 'systems'
providers_dir = PROJECT_ROOT / 'configs' / 'providers'

system_docs = {}
for path in systems_dir.glob('*.yaml'):
    system_docs.update(read_yaml(path).get('systems', {}))

selected_systems = SYSTEMS or experiment.get('systems', [])
selected_providers = PROVIDERS or experiment.get('providers', [])
provider_docs = {provider: read_yaml(providers_dir / f'{provider}.yaml') for provider in selected_providers}

experiment_summary = {
    'project_root': str(PROJECT_ROOT),
    'experiment_file': str(EXPERIMENT_FILE.relative_to(PROJECT_ROOT)),
    'experiment_run_id': EXPERIMENT_RUN_ID,
    'experiment_id': experiment.get('id'),
    'dataset_profile': str(dataset_profile.relative_to(PROJECT_ROOT)),
    'dataset_paths': {name: value.get('path') for name, value in dataset_doc.get('datasets', {}).items()},
    'systems': {
        system_id: {
            'label': system_docs[system_id].get('label'),
            'runtime_kind': system_docs[system_id].get('runtime_kind'),
            'modules': system_docs[system_id].get('modules'),
        }
        for system_id in selected_systems
    },
    'providers': {
        provider_id: {
            'provider': provider_docs[provider_id].get('llm', {}).get('provider'),
            'model': provider_docs[provider_id].get('llm', {}).get('model'),
            'temperature': provider_docs[provider_id].get('llm', {}).get('temperature'),
            'api_key_env': provider_docs[provider_id].get('llm', {}).get('api_key_env'),
        }
        for provider_id in selected_providers
    },
    'max_samples': MAX_SAMPLES,
}
experiment_summary

In [ ]:
def run_command(args):
    print(' '.join(str(item) for item in args))
    completed = subprocess.run(args, cwd=str(PROJECT_ROOT), text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    completed.check_returncode()
    return completed


def experiment_args(phase, extra_args=None):
    args = [
        sys.executable,
        'evaluation/run_experiment.py',
        '--experiment', str(EXPERIMENT_FILE),
        '--phase', phase,
        '--experiment-run-id', EXPERIMENT_RUN_ID,
    ]
    if SYSTEMS:
        args += ['--systems', *SYSTEMS]
    if PROVIDERS:
        args += ['--providers', *PROVIDERS]
    if MAX_SAMPLES is not None:
        args += ['--max-samples', str(MAX_SAMPLES)]
    if extra_args:
        args += list(extra_args)
    return args

## 0.1 Command Preview

In [ ]:
{
    'runtime_command': ' '.join(str(item) for item in experiment_args('runtime', ['--rerun-api-429'])),
    'evaluate_command': ' '.join(str(item) for item in experiment_args('evaluate')),
}

## 1. Phase RUNTIME

Chay TrustedSQL va ghi raw runtime output.

In [ ]:
run_command(experiment_args('runtime', ['--rerun-api-429']))

## 2. Phase EVALUATE

Tinh automatic metrics. Khong co human review/adjudication phase.

In [ ]:
run_command(experiment_args('evaluate'))

## 3. Load Metrics

In [ ]:
import csv

with (EXPERIMENT_DIR / 'run_index.csv').open('r', encoding='utf-8-sig', newline='') as handle:
    run_rows = list(csv.DictReader(handle))
RUN_ID = run_rows[0]['run_id']
RUN_DIR = PROJECT_ROOT / 'outputs' / 'runs' / RUN_ID
if not RUN_DIR.exists():
    RUN_DIR = PROJECT_ROOT / 'outputs' / 'evaluation' / RUN_ID
METRICS_DIR = RUN_DIR / 'evaluation' / 'metrics'
with (METRICS_DIR / 'benchmark_metrics.json').open('r', encoding='utf-8-sig') as handle:
    metrics = json.load(handle)
run_rows, RUN_ID, metrics.keys()

## 3.1 Runtime Config Snapshot

In [ ]:
def load_runtime_config_snapshot(run_id):
    run_dir = PROJECT_ROOT / 'outputs' / 'runs' / run_id
    manifest_path = run_dir / 'run_manifest.json'
    legacy_method_snapshot = PROJECT_ROOT / 'outputs' / 'evaluation' / run_id / 'run_config_snapshot.json'
    legacy_architecture_manifest = PROJECT_ROOT / 'outputs' / 'evaluation' / run_id / 'run_manifest.json'
    if manifest_path.exists():
        with manifest_path.open('r', encoding='utf-8-sig') as handle:
            snapshot = json.load(handle)
        runtime_kind = snapshot.get('runtime_kind') or ('architecture_baseline' if snapshot.get('architectures') else 'trustedsql')
        if runtime_kind == 'trustedsql':
            return {
                'run_id': run_id,
                'runtime_kind': runtime_kind,
                'settings': snapshot.get('resolved_config', {}).get('settings', {}),
                'module_models': snapshot.get('module_models', {}),
                'dataset_turn_count': snapshot.get('benchmark_selection', {}).get('turn_count'),
                'sequence_count': snapshot.get('benchmark_selection', {}).get('sequence_count'),
            }
        return {
            'run_id': run_id,
            'runtime_kind': runtime_kind,
            'architectures': snapshot.get('architectures'),
            'llm': snapshot.get('config', {}).get('llm'),
            'dataset_counts': snapshot.get('dataset_counts'),
        }
    if legacy_method_snapshot.exists():
        with legacy_method_snapshot.open('r', encoding='utf-8-sig') as handle:
            snapshot = json.load(handle)
        return {
            'run_id': run_id,
            'runtime_kind': 'trustedsql',
            'settings': snapshot.get('resolved_config', {}).get('settings', {}),
            'module_models': snapshot.get('module_models', {}),
            'dataset_turn_count': snapshot.get('benchmark_selection', {}).get('turn_count'),
            'sequence_count': snapshot.get('benchmark_selection', {}).get('sequence_count'),
        }
    if legacy_architecture_manifest.exists():
        with legacy_architecture_manifest.open('r', encoding='utf-8-sig') as handle:
            manifest = json.load(handle)
        return {
            'run_id': run_id,
            'runtime_kind': 'architecture_baseline',
            'architectures': manifest.get('architectures'),
            'llm': manifest.get('config', {}).get('llm'),
            'dataset_counts': manifest.get('dataset_counts'),
        }
    return {'run_id': run_id, 'error': 'missing runtime snapshot'}


runtime_config_summaries = [load_runtime_config_snapshot(row['run_id']) for row in run_rows]
runtime_config_summaries

## 4. Utility Metrics

In [ ]:
metrics['utility']

## 5. Single-turn Security Metrics

In [ ]:
metrics['single_turn_security']

## 6. Multi-turn Security Metrics

In [ ]:
metrics['multi_turn_security']

## 7. Performance Metrics

In [ ]:
metrics['performance']

## 8. Output Files

In [ ]:
sorted(path.name for path in METRICS_DIR.iterdir())